# Activations & Gradients / BatchNorm —— 生产案例集（Part 3 专属）

本文件是配合 `Activations_&_Gradients,_BatchNorm.ipynb`（注释版）的**可直接运行**版本。
和注释版不同，这里的每段代码都是**我重新写的、干净可跑、绝不会卡死电脑**的（步数都很小，几秒钟出结果）。

聚焦 Part 3 的核心工程问题——**「怎么让网络内部保持健康」**：

| 案例 | 解决的生产问题 |
|---|---|
| 案例 1 | **初始化诊断**：为什么第一步 loss 应该 ≈ ln(27)，而不是几十？(hockey-stick 曲线) |
| 案例 2 | **激活饱和 / 死神经元**：tanh 增益选多大？怎么用「白条图」一眼看出饱和 |
| 案例 3 | **深层方差漂移**：不归一化时激活 std 会随层数爆炸/消失，BatchNorm 如何把它按住 |
| 案例 4 | **update:data 比例**：一种比「盯 loss」更靠谱的学习率体检法 |
| 案例 5 | **BN 训练/推理模式陷阱** + 保存/加载：忘了切 eval 模式会让线上指标翻车 |

> 运行顺序：从上到下依次运行即可。**案例 0 必须先跑**（准备数据和公共组件）。

## 案例 0：公共准备（数据 + 可复用的层模块）

读数据、建词表、切分训练/验证集，并把 `Linear / BatchNorm1d / Tanh` 封装成**可复用模块**（和讲座里一致，但每个层都显式接收随机数发生器 `g`，更适合生产里复现实验）。

In [ ]:
import torch                       # 主库
import torch.nn.functional as F     # cross_entropy 等
import matplotlib.pyplot as plt     # 画图
import random

# 让 matplotlib 能显示中文(macOS 自带 'Arial Unicode MS';没有就退回英文也不影响运行)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS','Heiti SC','Songti SC','Hiragino Sans GB','sans-serif']
plt.rcParams['axes.unicode_minus'] = False   # 负号正常显示

# ---- 读数据 + 词表 ----
words = open('names.txt','r').read().splitlines()
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}; stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
block_size = 3

def build_dataset(words):
    X,Y=[],[]
    for w in words:
        ctx=[0]*block_size
        for ch in w+'.':
            ix=stoi[ch]; X.append(ctx); Y.append(ix); ctx=ctx[1:]+[ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42); random.shuffle(words)
n1=int(0.8*len(words)); n2=int(0.9*len(words))
Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])
print('训练集:', tuple(Xtr.shape), ' 验证集:', tuple(Xdev.shape), ' 词表:', vocab_size)

In [ ]:
# ---- 可复用的层模块(生产写法:显式传入 generator g,方便复现) ----
class Linear:
    def __init__(self, fan_in, fan_out, g, bias=True):
        self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5  # Kaiming: 除 sqrt(fan_in)
        self.bias = torch.zeros(fan_out) if bias else None
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None: self.out = self.out + self.bias
        return self.out
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps, self.momentum, self.training = eps, momentum, True
        self.gamma = torch.ones(dim); self.beta = torch.zeros(dim)          # 可训练
        self.running_mean = torch.zeros(dim); self.running_var = torch.ones(dim)  # 缓冲区
    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True); xvar = x.var(0, keepdim=True)  # 用当前 batch
        else:
            xmean = self.running_mean; xvar = self.running_var             # 推理用滑动统计
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():                                          # 注意:这里是 with,不是 while!
                self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*xmean
                self.running_var  = (1-self.momentum)*self.running_var  + self.momentum*xvar
        return self.out
    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x): self.out = torch.tanh(x); return self.out
    def parameters(self): return []

print('层模块已就绪:Linear / BatchNorm1d / Tanh')

## 案例 1：初始化诊断 —— 第一步 loss 应该是多少？

**生产问题**：训练一开始 loss 高得离谱、然后猛地掉下来（loss 曲线像曲棍球杆 hockey-stick），
说明**初始化没做好**——最开始那几百步都在「把过大的 logits 压回接近 0」，纯属浪费。

**理论值**：27 个字符若初始等概率，交叉熵 ≈ `ln(27) ≈ 3.30`。
下面对比「坏初始化(输出层不缩放)」和「好初始化(输出层 ×0.01)」的**第一步 loss** 和**前 200 步曲线**。

In [ ]:
import math
print(f'理论初始 loss ≈ ln(27) = {math.log(vocab_size):.3f}\n')

def make_model(g, scale_last):
    # 一个 2 隐藏层的小 MLP;scale_last 控制输出层权重缩放
    n_embd, n_hidden = 10, 100
    C = torch.randn((vocab_size, n_embd), generator=g)
    layers = [
        Linear(n_embd*block_size, n_hidden, g), Tanh(),
        Linear(n_hidden, n_hidden, g), Tanh(),
        Linear(n_hidden, vocab_size, g),
    ]
    with torch.no_grad():
        layers[-1].weight *= scale_last          # 关键:缩放输出层
    params = [C] + [p for l in layers for p in l.parameters()]
    for p in params: p.requires_grad = True
    return C, layers, params

def first_step_loss(scale_last):
    g = torch.Generator().manual_seed(2147483647)
    C, layers, _ = make_model(g, scale_last)
    Xb, Yb = Xtr[:32], Ytr[:32]
    x = C[Xb].view(32, -1)
    for l in layers: x = l(x)
    return F.cross_entropy(x, Yb).item()

print(f'坏初始化 (输出层 ×1.0 ) 第一步 loss = {first_step_loss(1.0):.3f}   ← 远大于 3.30,开局就在做无用功')
print(f'好初始化 (输出层 ×0.01) 第一步 loss = {first_step_loss(0.01):.3f}   ← 贴近理论值,开局就健康')

In [ ]:
# 训练前 200 步,对比两种初始化的 loss 曲线(200 步很快,不会卡)
seeds=2147483647
def train_curve(scale_last, steps=200):
    g = torch.Generator().manual_seed(seeds)
    C, layers, params = make_model(g, scale_last)
    losses = []
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
        Xb, Yb = Xtr[ix], Ytr[ix]
        x = C[Xb].view(32, -1)
        for l in layers: x = l(x)
        loss = F.cross_entropy(x, Yb)
        for p in params: p.grad = None
        loss.backward()
        for p in params: p.data += -0.1 * p.grad
        losses.append(loss.item())
    return losses

plt.figure(figsize=(10,4))
plt.plot(train_curve(1.0),  label='坏初始化 ×1.0 (曲棍球杆)')
plt.plot(train_curve(0.01), label='好初始化 ×0.01 (开局就低)')
plt.axhline(math.log(vocab_size), color='k', ls='--', lw=1, label='理论值 ln(27)')
plt.xlabel('step'); plt.ylabel('loss'); plt.legend(); plt.title('初始化对开局 loss 的影响');

## 案例 2：激活饱和 & 死神经元 —— tanh 增益该选多大？

**生产问题**：tanh 层若大量神经元输出贴近 ±1，就**饱和**了——梯度≈0，这些神经元**学不动**（近似「死」了）。
`Linear` 后接 tanh 时常乘一个**增益 gain**（tanh 的理论最优约 `5/3≈1.67`）。gain 太大 → 饱和；太小 → 激活缩成一团、表达力弱。

下面：(1) 扫不同 gain，打印饱和比例；(2) 画 Karpathy 那张著名的**「白条图」**——白色=某样本某神经元饱和。

In [ ]:
@torch.no_grad()
def saturation_for_gain(gain, n_hidden=200):
    g = torch.Generator().manual_seed(2147483647)
    C = torch.randn((vocab_size, 10), generator=g)
    W = torch.randn((10*block_size, n_hidden), generator=g) / (10*block_size)**0.5 * gain
    Xb = Xtr[:1000]                        # 拿 1000 个样本看统计
    h = torch.tanh(C[Xb].view(1000, -1) @ W)
    sat = (h.abs() > 0.97).float().mean().item() * 100
    return h, sat

print('gain     激活std    饱和(|h|>0.97)比例')
for gain in [0.5, 1.0, 5/3, 3.0]:
    h, sat = saturation_for_gain(gain)
    print(f'{gain:4.2f}     {h.std():.3f}      {sat:5.1f}%')
print('\n经验:gain=5/3 附近饱和比例适中;gain=3 大量饱和(很多神经元学不动)')

In [ ]:
# 「白条图」:白色像素 = 该(样本, 神经元)饱和。竖直白条 = 某神经元对所有样本都饱和 = 死神经元
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, gain in zip(axes, [5/3, 3.0]):
    h, sat = saturation_for_gain(gain)
    ax.imshow((h.abs() > 0.97).numpy(), cmap='gray', aspect='auto')
    ax.set_title(f'gain={gain:.2f}  饱和 {sat:.1f}%   (白=饱和)')
    ax.set_xlabel('神经元'); ax.set_ylabel('样本')
plt.tight_layout();

## 案例 3：深层网络的方差漂移 —— BatchNorm 为什么关键

**生产问题**：网络一深，前向激活的 **std（标准差）会随层数指数漂移**——权重稍大就爆炸、稍小就消失，梯度跟着完蛋。
下面把一个随机向量过 **50 层线性层**，对比三种情况的「每层激活 std」：
- 权重不缩放（爆炸）、
- 权重 Kaiming 缩放（大致稳住，但仍会缓慢漂移）、
- 每层后加 **BatchNorm**（**牢牢按在 1 附近**）。

In [ ]:
@torch.no_grad()
def depth_std(mode, depth=50, n=256):
    g = torch.Generator().manual_seed(0)
    x = torch.randn((n, n), generator=g)          # 一批向量 (batch=n, dim=n)
    bn = BatchNorm1d(n)
    stds = []
    for _ in range(depth):
        w = torch.randn((n, n), generator=g)
        if mode == 'kaiming' or mode == 'bn':
            w = w / n**0.5                          # Kaiming 缩放
        x = x @ w
        if mode == 'bn':
            x = bn(x)                               # 每层后归一化
        x = torch.tanh(x)
        stds.append(x.std().item())
    return stds

plt.figure(figsize=(10,4))
plt.plot(depth_std('none'),    label='不缩放 (爆炸/坍缩)')
plt.plot(depth_std('kaiming'), label='Kaiming 缩放 (较稳)')
plt.plot(depth_std('bn'),      label='+ BatchNorm (锁在 ~1)')
plt.xlabel('第几层'); plt.ylabel('该层激活 std'); plt.yscale('log')
plt.legend(); plt.title('50 层深网络:激活 std 随深度的变化');

## 案例 4：用 update:data 比例给学习率「体检」

**生产问题**：光盯 loss 曲线很难判断学习率好坏。更靠谱的指标是每步的
**update:data 比例 = std(lr·grad) / std(param)** 的 log10 —— 经验上健康值约 **1e-3（即 -3）**。
太高=学习率过大（不稳），太低=过小（学得太慢）。下面对比 lr=0.001 与 lr=0.1。

In [ ]:
def train_track_ud(lr, steps=800):
    g = torch.Generator().manual_seed(2147483647)
    n_embd, n_hidden = 10, 100
    C = torch.randn((vocab_size, n_embd), generator=g)
    layers = [
        Linear(n_embd*block_size, n_hidden, g, bias=False), BatchNorm1d(n_hidden), Tanh(),
        Linear(n_hidden, n_hidden, g, bias=False), BatchNorm1d(n_hidden), Tanh(),
        Linear(n_hidden, vocab_size, g, bias=False), BatchNorm1d(vocab_size),
    ]
    with torch.no_grad(): layers[-1].gamma *= 0.1
    params = [C] + [p for l in layers for p in l.parameters()]
    for p in params: p.requires_grad = True
    ud = []
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
        Xb, Yb = Xtr[ix], Ytr[ix]
        x = C[Xb].view(32, -1)
        for l in layers: x = l(x)
        loss = F.cross_entropy(x, Yb)
        for p in params: p.grad = None
        loss.backward()
        for p in params: p.data += -lr * p.grad
        with torch.no_grad():
            ud.append([((lr*p.grad).std()/p.data.std()).log10().item()
                       for p in params if p.ndim==2])
    return ud

fig, axes = plt.subplots(1, 2, figsize=(14,4), sharey=True)
for ax, lr in zip(axes, [0.001, 0.1]):
    ud = train_track_ud(lr)
    for k in range(len(ud[0])):
        ax.plot([row[k] for row in ud])
    ax.axhline(-3, color='k', ls='--', label='理想 ~1e-3')
    ax.set_title(f'lr={lr}'); ax.set_xlabel('step'); ax.legend()
axes[0].set_ylabel('log10(update:data)')
fig.suptitle('lr=0.001 曲线远低于 -3(太慢)  vs  lr=0.1 贴近 -3(合适)');

## 案例 5：BatchNorm 的训练/推理模式陷阱 + 保存/加载

**生产事故常见款**：训练用了 BatchNorm，评估/上线时**忘了切到 eval 模式**，于是推理仍用「当前 batch 的统计量」。
如果线上是**逐条推理（batch=1）**，BN 现算的 std 会出问题，指标直接翻车。
下面训练一个带 BN 的小模型，然后对比 `training=True`(错) 与 `training=False`(对) 的验证 loss，并演示**保存/加载**。

In [ ]:
# 训练一个带 BN 的小模型(1500 步,安全)
g = torch.Generator().manual_seed(2147483647)
n_embd, n_hidden = 10, 100
C = torch.randn((vocab_size, n_embd), generator=g)
layers = [
    Linear(n_embd*block_size, n_hidden, g, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size, g, bias=False), BatchNorm1d(vocab_size),
]
with torch.no_grad(): layers[-1].gamma *= 0.1
params = [C] + [p for l in layers for p in l.parameters()]
for p in params: p.requires_grad = True

for i in range(1500):
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]
    x = C[Xb].view(32, -1)
    for l in layers: x = l(x)
    loss = F.cross_entropy(x, Yb)
    for p in params: p.grad = None
    loss.backward()
    for p in params: p.data += -0.1 * p.grad
print(f'训练最后一步 loss = {loss.item():.3f}')

In [ ]:
@torch.no_grad()
def eval_loss(X, Y):
    x = C[X].view(X.shape[0], -1)
    for l in layers: x = l(x)
    return F.cross_entropy(x, Y).item()

# 错误:忘了切 eval,BN 用的是「整个验证集这一大批」的统计量
for l in layers: l.training = True
wrong = eval_loss(Xdev, Ydev)

# 正确:切到 eval,BN 用训练期间累积的 running_mean/var
for l in layers: l.training = False
right = eval_loss(Xdev, Ydev)

print(f'training=True  (忘切/错) 验证 loss = {wrong:.4f}')
print(f'training=False (正确)    验证 loss = {right:.4f}')
print('\n注:整批评估时两者接近;但线上若 batch=1 逐条推理,training=True 会直接出错/失真——务必切 eval!')

In [ ]:
# 保存 / 加载(生产里存的是:参数 + 词表 + 超参 + BN 的 running 缓冲区)
ckpt = {
    'C': C,
    'layer_params': [ [t.detach() for t in l.parameters()] for l in layers ],
    'bn_buffers': [ (l.running_mean, l.running_var) for l in layers if isinstance(l, BatchNorm1d) ],
    'stoi': stoi, 'itos': itos, 'block_size': block_size,
}
torch.save(ckpt, 'bn_demo.pt')
loaded = torch.load('bn_demo.pt', weights_only=False)
print('已保存并重新加载 bn_demo.pt ✓   词表大小:', len(loaded['itos']),
      ' | 保存了', len(loaded['bn_buffers']), '组 BN 缓冲区(running_mean/var)')

## 小结

Part 3 的工程主线是**「让网络内部保持健康」**，本案例集对应五个抓手：
1. **初始化**：缩小输出层，让第一步 loss 就 ≈ ln(vocab)，别浪费开局（案例 1）。
2. **激活分布**：用饱和比例 + 白条图盯 tanh，别让神经元死掉；gain 选 5/3 附近（案例 2）。
3. **BatchNorm**：深层网络里把每层激活 std 按在 1 附近，方差不再随深度漂移（案例 3）。
4. **update:data 比例**：比盯 loss 更灵敏的学习率体检，健康值 ~1e-3（案例 4）。
5. **train/eval 模式 + 存盘**：BN 上线必须切 eval，并把 running 缓冲区一起存（案例 5）。

> 想看这些机制在**原始跟练代码**里怎么写的（含 bug 标注），回到 `Activations_&_Gradients,_BatchNorm.ipynb`。